# 📊 Prometheus & Grafana Monitoring Dashboard
This notebook serves as the management console for infrastructure and model monitoring of your fraud detection project.

In [ ]:
import os
import requests
from IPython.display import display, HTML

# ==========================================
# STEP 1: Configuration & Endpoints
# ==========================================
print("=== Prometheus & Grafana Infrastructure Check ===")

PROMETHEUS_URL = "http://localhost:9090"
GRAFANA_URL = "http://localhost:3000"

# ==========================================
# STEP 2: Status Check (Health Checks)
# ==========================================
def check_service(name, url):
    try:
        response = requests.get(url, timeout=3)
        if response.status_code < 400:
            print(f"✅ {name} is reachable at: {url}")
            return True
        else:
            print(f"⚠️ {name} responded with status code: {response.status_code}")
            return False
    except requests.ConnectionError:
        print(f"❌ {name} is NOT reachable ({url}). Is the service running?")
        return False

print("\n--- Checking running services ---")
prom_status = check_service("Prometheus", PROMETHEUS_URL)
grafana_status = check_service("Grafana", GRAFANA_URL)

# ==========================================
# STEP 3: GPU Metrics Check (DCGM Exporter)
# ==========================================
if prom_status:
    print("\n--- Checking Prometheus GPU metrics ---")
    try:
        query_url = f"{PROMETHEUS_URL}/api/v1/query?query=up"
        res = requests.get(query_url, timeout=3).json()
        
        targets = res.get("data", {}).get("result", [])
        print(f"📊 Active Prometheus targets found: {len(targets)}")
        for target in targets:
            job = target.get("metric", {}).get("job", "unknown")
            health = target.get("value", ["", "unknown"])[1]
            print(f"   - Job '{job}': Status {health}")
    except Exception as e:
        print(f"⚠️ Could not query Prometheus metrics: {e}")

# ==========================================
# STEP 4: Link Useful Dashboards
# ==========================================
print("\n--- Dashboard links ---")
display(HTML(f"""
<div style="padding: 15px; border: 1px solid #4CAF50; border-radius: 5px; background-color: #f9f9f9;">
    <h3>🔗 Quick Access to Your Monitoring Interfaces</h3>
    <ul>
        <li><b>Grafana Web-UI:</b> <a href="{GRAFANA_URL}" target="_blank">{GRAFANA_URL}</a> (Ideal for visualizing loss, GPU utilization & VRAM)</li>
        <li><b>Prometheus Web-UI:</b> <a href="{PROMETHEUS_URL}" target="_blank">{PROMETHEUS_URL}</a> (For direct PromQL queries)</li>
    </ul>
    <p><i>Tip: If the links do not open, make sure the ports have been forwarded via SSH port forwarding (e.g., <code>-L 3000:localhost:3000 -L 9090:localhost:9090</code>).</i></p>
</div>
"""))